# Topoplots, selected faithfulness and spectra

Set `DEBUG = True` in the setup cell to write figures and derived CSVs under `Temp`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Daprosero/STF-KernelSHAP.git"
REPO_NAME = "STF-KernelSHAP"
PACKAGE_PATH = Path("src") / "stf_kernelshap"


def run_command(command, cwd=None):
    result = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        message = [
            f"Command failed: {' '.join(map(str, command))}",
            f"Return code: {result.returncode}",
        ]
        if result.stdout:
            message.append(f"STDOUT:\n{result.stdout}")
        if result.stderr:
            message.append(f"STDERR:\n{result.stderr}")
        raise RuntimeError("\n".join(message))
    return result


def resolve_working_root():
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working")
    if Path("/content").exists():
        return Path("/content")
    return Path.cwd().resolve()


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / PACKAGE_PATH).exists():
            return candidate

    working_root = resolve_working_root()
    clone_dir = working_root / REPO_NAME

    if (clone_dir / PACKAGE_PATH).exists():
        return clone_dir.resolve()

    if clone_dir.exists() and (clone_dir / ".git").exists():
        run_command(["git", "-C", str(clone_dir), "pull", "--ff-only"])
        if (clone_dir / PACKAGE_PATH).exists():
            return clone_dir.resolve()

    if clone_dir.exists() and not (clone_dir / PACKAGE_PATH).exists():
        clone_dir = working_root / f"{REPO_NAME}_repo"

    if not clone_dir.exists():
        run_command(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)])

    if not (clone_dir / PACKAGE_PATH).exists():
        raise FileNotFoundError(
            f"El repositorio clonado no contiene {PACKAGE_PATH}: {clone_dir}"
        )

    return clone_dir.resolve()


def install_project_requirements(project_root):
    requirements_path = project_root / "requirements.txt"
    if not requirements_path.exists():
        raise FileNotFoundError(f"No existe requirements.txt en {project_root}")
    run_command([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(requirements_path),
    ])


PROJECT_ROOT = resolve_project_root()
os.chdir(PROJECT_ROOT)

RUNNING_IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
RUNNING_IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()
INSTALL_REQUIREMENTS = RUNNING_IN_COLAB or RUNNING_IN_KAGGLE

if INSTALL_REQUIREMENTS:
    install_project_requirements(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INSTALL_REQUIREMENTS:", INSTALL_REQUIREMENTS)


In [ ]:
from stf_kernelshap.notebook_setup import setup_notebook_environment

DEBUG = True
paths = setup_notebook_environment(debug=DEBUG)

DATA_DIR = paths.data_dir
MODELS_DIR = paths.models_dir
OUTPUT_MODELS_DIR = paths.output_models_dir
RESULTS_DIR = paths.results_dir
FIGURES_DIR = paths.figures_dir


In [ ]:
from stf_kernelshap.visualization.topoplots import (
    MI_07_SUBJECTS_TO_EXTRACT,
    MI_CHANNELS,
    MI_FREQ_BANDS,
    MI_FREQ_LABELS,
    MI_SUBJECTS_TO_EXTRACT,
    TDAH_CHANNELS,
    TDAH_DATA_FOLD,
    TDAH_FREQ_BANDS,
    TDAH_FREQ_LABELS,
    build_fidelity_auc_per_sample_table,
    compute_mean_psd_welch,
    load_mi_case,
    load_selected_mi_07_segments,
    load_selected_tdah_segments,
    plot_methods_rows_x6_topomap_difference,
    plot_selected_faithfulness_figures,
    plot_stf_single_subject_grid,
    plot_stf_two_subjects_vertical_2x4,
    run_selected_faithfulness_all,
)


In [ ]:
# Example topomap inputs. Adjust these before running.
mi_case_1 = load_mi_case(
    window_name="2.5-5",
    subject_id=43,
    fold=1,
    results_dir=str(RESULTS_DIR),
    base_path_mi=str(DATA_DIR / "MI"),
)
mi_case_2 = load_mi_case(
    window_name="0-7",
    subject_id=14,
    fold=4,
    results_dir=str(RESULTS_DIR),
    base_path_mi=str(DATA_DIR / "MI"),
)

plot_methods_rows_x6_topomap_difference(
    cases=[mi_case_1, mi_case_2],
    channels=MI_CHANNELS,
    save_path=str(FIGURES_DIR / "MI_topoplots.pdf"),
)


In [ ]:
# Selected faithfulness workflow.
df_per_sample, df_plot_ready, output_paths = run_selected_faithfulness_all(
    recompute_metrics=True,
    results_dir=str(RESULTS_DIR),
    output_dir=str(RESULTS_DIR / "faithfulness_selected_relevance"),
    figures_dir=str(FIGURES_DIR),
    base_path_mi=str(DATA_DIR / "MI"),
    folds_path=str(DATA_DIR / "TDAH" / "folds.pkl"),
    path_adhd=str(DATA_DIR / "TDAH" / "ieee" / "ADHD_group"),
    path_control=str(DATA_DIR / "TDAH" / "ieee" / "Control_group"),
    models_tdah_root=str(MODELS_DIR / "TDAH"),
)
plot_selected_faithfulness_figures(df_plot_ready, output_dir=str(FIGURES_DIR))
df_auc_per_sample = build_fidelity_auc_per_sample_table(
    df_per_sample,
    output_dir=str(RESULTS_DIR / "faithfulness_selected_relevance"),
)


In [ ]:
# Average PSD figure inputs. Plotting remains explicit in the notebook.
X_mi_07_selected = load_selected_mi_07_segments(
    base_path_mi=str(DATA_DIR / "MI"),
    mi_subjects_to_extract=MI_07_SUBJECTS_TO_EXTRACT,
)
X_tdah_selected = load_selected_tdah_segments(
    fold_to_extract=TDAH_DATA_FOLD,
    folds_path=str(DATA_DIR / "TDAH" / "folds.pkl"),
    path_adhd=str(DATA_DIR / "TDAH" / "ieee" / "ADHD_group"),
    path_control=str(DATA_DIR / "TDAH" / "ieee" / "Control_group"),
)

freqs_mi, psd_mi_mean, _ = compute_mean_psd_welch(X_mi_07_selected, fs=128, nperseg=256)
freqs_tdah, psd_tdah_mean, _ = compute_mean_psd_welch(X_tdah_selected, fs=128, nperseg=256)
